# ADSC 4050 Project 
## S&P500 Clustering and Volatility Forecast
Date: Friday 17, 2026

Prepared by:
1. Mint Thai - T00762325
2. Cynthia Urrutia - T00735328

# Introduction

Financial markets exhibit complex and heterogeneous behavior; however, patterns and commonalities can still be identified across assets. Stocks can often be categorized into behavioral groups such as defensive and cyclical, reflecting differences in stability and responsiveness to market conditions.

This project approaches the analysis through both unsupervised learning and predictive modeling techniques. Specifically, K-means clustering is applied to identify groups of stocks with similar characteristics. Subsequently, it is investigated whether such grouping can enhance the prediction of future stock volatility.

The dataset used in this study consists of daily observations from S&P 500 companies over a five-year period. It includes variables such as stock prices, trading volume, sector classification, and index weights. It is also important to note that certain sectors, such as technology and financials, represent a larger portion of the index, while others, such as energy and utilities, contribute less, reflecting differences in market composition.

To ensure consistency across the project, the same time-based split is used for both clustering and forecasting: training (March 2021 to March 2024), validation (March 2024 to March 2025), and test (March 2025 to March 2026). In addition, only stocks that are present across all three periods are retained, ensuring a balanced panel and comparability of results across the analysis.

Although both analyses rely on the same data partition, the feature engineering differs according to the objective of each task. For clustering, the features summarize stock behavior through returns, rolling volatility, and volume changes over 1-day, 5-day, and 21-day horizons. For forecasting, the features are constructed to predict future 21-day volatility and therefore include recent returns, rolling standard deviations, intraday price range, relative trading volume, and cluster-based indicators.

The objective of this project is to identify meaningful stock clusters and evaluate whether incorporating this grouping information improves the accuracy of volatility forecasting. By combining these techniques, this study aims to provide deeper insight into market structure and improve the understanding of volatility dynamics.

# Research Questions and Objectives

The primary research question of this study is:

**Does clustering stocks based on their return, volatility, and volume characteristics improve the accuracy of stock volatility forecasting?**

Volatility plays a central role in financial markets, as it reflects the magnitude and speed of price fluctuations over time. Higher volatility is associated with larger and less predictable price movements, indicating greater uncertainty and risk. As a result, accurate volatility forecasting is essential for investment decision-making, portfolio allocation, and risk management. Improving volatility forecasts can therefore enhance the understanding of market dynamics and support more informed financial strategies.

To address this research question, the analysis is structured into two main components: (1) unsupervised clustering of stocks and (2) supervised volatility forecasting. The clustering step aims to identify groups of stocks with similar statistical behavior, while the forecasting step evaluates whether incorporating this grouping information improves predictive performance.

For the clustering component, K-means clustering is applied using features that capture stock behavior in terms of returns, volatility, and trading activity across multiple time horizons. Three feature combinations are explored to assess how temporal structure influences clustering results, capturing immediate (1-day), short-term (5-day), and medium-term (21-day) market dynamics. The resulting clusters are intended to represent distinct patterns of market behavior, such as lower-volatility and higher-volatility stocks.

Building on the clustering results, a forecasting framework is implemented to estimate future 21-day stock volatility. The primary model used is a Long Short-Term Memory (LSTM) neural network, which is well-suited for capturing temporal dependencies in financial time series data. The model incorporates features that reflect recent price movements, historical volatility, intraday price behavior, and trading activity. In addition, information derived from the clustering step is included as an explanatory input to assess its contribution to forecasting performance.

A naïve baseline model, defined as the most recently observed volatility, is used as a benchmark for comparison. Model performance is evaluated using standard regression metrics, including Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and the coefficient of determination (R²).

The overall objective of this study is to determine whether clustering provides meaningful structure in financial data and whether this structure can be leveraged to improve volatility forecasting accuracy. By combining unsupervised learning with time-series prediction techniques, the analysis aims to provide insights into both market segmentation and predictive performance.

# Model Construction

This section describes the construction of the clustering and forecasting models, including feature engineering, mathematical definitions of variables, and the rationale behind the selected methodologies.

K-means clustering is selected for its simplicity, consistency, and interpretability in identifying groups of stocks with similar behavior. Compared to alternative methods, such as hierarchical clustering, K-means provided more stable and consistent groupings. In addition, other forecasting techniques, including Random Forest models, were explored but yielded lower predictive performance, and are therefore not the focus of this analysis.

For the forecasting task, a Long Short-Term Memory (LSTM) neural network is employed due to its ability to capture nonlinear temporal dependencies in financial time series data. A naïve baseline model is also included to provide a benchmark, allowing evaluation of whether the proposed models extract additional predictive information beyond volatility persistence.

Model performance is evaluated using multiple metrics. For clustering, the silhouette score and elbow method are used to assess cluster quality and determine the optimal number of clusters. For forecasting, Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and the coefficient of determination ($R^2$) are used to measure prediction accuracy and explanatory power on unseen data.

The dataset is structured as panel data indexed by stock symbol and date. Observations are sorted chronologically within each stock to preserve the temporal structure of the series. A time-based split is applied: training (March 2021 to March 2024), validation (March 2024 to March 2025), and test (March 2025 to March 2026). Only stocks present in all three periods are retained to ensure a balanced panel and comparability across models.

Although both clustering and forecasting use the same data partition, feature construction differs according to the objective of each task.

## Clustering Feature Construction

For the clustering component, features are derived from closing prices and trading volume. Let $P_{i,t}$ denote the closing price of stock $i$ at time $t$, and $V_{i,t}$ its trading volume.

Feature construction focuses on three key dimensions of stock behavior: returns, risk, and trading activity.

- **Returns:** To stabilize variance and improve comparability across stocks, prices are transformed using the logarithmic function, defined as $\text{log\_close}_{i,t} = \log(P_{i,t})$. Log returns over different horizons $k \in \{1,5,21\}$ are then computed as $r^{(k)}_{i,t} = \log(P_{i,t}) - \log(P_{i,t-k})$.

- **Risk (Volatility):** Volatility is measured as the rolling standard deviation of daily log returns, defined as $\sigma^{(k)}_{i,t} = \mathrm{sd}(r^{(1)}_{i,t-k+1}, \dots, r^{(1)}_{i,t})$, where $\mathrm{sd}(\cdot)$ denotes the sample standard deviation.

- **Trading Activity:** To reduce skewness in trading activity, volume is transformed using $\text{log\_volume}_{i,t} = \log(\max(V_{i,t},1))$. Changes in trading activity are then computed as $\Delta v^{(k)}_{i,t} = \text{log\_volume}_{i,t} - \text{log\_volume}_{i,t-k}$.

### Feature Cases

Three feature configurations are constructed using $k = 1, 5, 21$, capturing different temporal dynamics. These correspond to immediate (1-day), short-term (5-day), and medium-term (21-day) horizons, allowing the clustering model to distinguish between short-lived fluctuations and more persistent patterns in stock behavior.

### Aggregation

Since clustering is performed at the stock level, features are aggregated over time. For each variable $x_{i,t}$, the mean and standard deviation are computed as:

- Mean: $\bar{x}_i = \frac{1}{T_i} \sum_{t=1}^{T_i} x_{i,t}$
- Standard deviation: $s_i(x) = \sqrt{\frac{1}{T_i - 1} \sum_{t=1}^{T_i} (x_{i,t} - \bar{x}_i)^2}$



## Forecasting Feature Construction

For the forecasting component, the objective is to predict future 21-day stock volatility based on recent market behavior. Let $C_{i,t}$ denote the closing price of stock $i$ at time $t$.

Daily returns are first computed as $R_{i,t} = \frac{C_{i,t} - C_{i,t-1}}{C_{i,t-1}}$, capturing the relative price change from one day to the next.

The target variable is defined as the future 21-day volatility, calculated as the standard deviation of returns over the next 21 trading days: $Y_{i,t} = \mathrm{sd}(R_{i,t+1}, R_{i,t+2}, \dots, R_{i,t+21})$. This represents the level of market uncertainty the model aims to predict.

Feature construction focuses on several key dimensions of market behavior: price dynamics, risk, intraday price behavior, trading activity, and structural information.

- **Price Dynamics:** The absolute return $|R_{i,t}|$ is included to capture the magnitude of the most recent price movement, reflecting short-term market shocks.

- **Risk (Volatility):** Rolling volatility measures are computed as $RV^{(k)}_{i,t} = \mathrm{sd}(R_{i,t-k+1}, \dots, R_{i,t})$ for $k \in \{5,10,21\}$. These variables capture volatility persistence over short- and medium-term horizons.

- **Intraday Price Behavior:** To account for within-day price variation, the high–low range is defined as $HL_{i,t} = \frac{H_{i,t} - L_{i,t}}{C_{i,t}}$, where $H_{i,t}$ and $L_{i,t}$ represent the daily high and low prices. Rolling averages are then computed as $\overline{HL}^{(k)}_{i,t} = \frac{1}{k} \sum_{j=0}^{k-1} HL_{i,t-j}$ for $k \in \{5,21\}$, capturing persistent intraday volatility.

- **Trading Activity:** Trading intensity is captured through relative volume. Average volume is first computed as $\overline{V}^{(k)}_{i,t} = \frac{1}{k} \sum_{j=0}^{k-1} V_{i,t-j}$, and relative volume is then defined as $RelVol^{(k)}_{i,t} = \frac{V_{i,t}}{\overline{V}^{(k)}_{i,t}}$. These variables measure deviations from typical trading activity.

- **Structural Information:** In addition to market-based predictors, a cluster-based indicator variable is included to account for differences between groups of stocks identified in the clustering stage.

## LSTM Forecasting Model

To model the temporal dynamics of volatility, a Long Short-Term Memory (LSTM) neural network is employed. This model processes sequences of past observations and learns patterns over time.

In general form, the LSTM updates its hidden state as $h_t = \mathrm{LSTM}(x_t, h_{t-1}, c_{t-1})$, where $x_t$ represents the input features at time $t$. The predicted volatility is then obtained as $\hat{Y}_{i,t} = W h_t + b$.

## Naïve Baseline Model

To benchmark performance, a naïve baseline is used. This approach assumes that future volatility is equal to the most recently observed 21-day volatility, defined as $\hat{Y}^{\text{naive}}_{i,t} = RV^{(21)}_{i,t}$.

# Results

This section presents the results of the clustering analysis and the volatility forecasting models. The findings are evaluated in relation to the research objective of determining whether clustering provides meaningful structure and improves forecasting performance.

## Clustering Results

The clustering analysis was conducted using three feature configurations based on different time horizons (1-day, 5-day, and 21-day). Among these, the 21-day feature configuration provided the most stable and interpretable results.

The optimal number of clusters was determined to be $K = 2$, based on both the silhouette score and the elbow method. The silhouette scores were 0.57 for the training set, 0.53 for the validation set, and 0.37 for the test set, indicating a reasonably well-defined clustering structure, although separation decreases slightly out-of-sample. This decline may be explained by changes in market behavior toward the end of the sample period. 

The resulting clusters are unbalanced, with Cluster 0 containing 448 stocks and Cluster 1 containing 51 stocks. Despite this imbalance, the clusters exhibit clear differences in behavior.

A more detailed examination of cluster-level statistics further highlights the differences between the two groups:

- Cluster 0 (lower-volatility group) exhibits positive average returns, with $\text{logret\_21} \approx 0.007$, and relatively low volatility, with $\text{logvol\_21} \approx 0.017$. 
- Cluster 1 (higher-volatility group) shows slightly negative average returns ($\text{logret\_21} \approx -0.001$) and significantly higher volatility levels ($\text{logvol\_21} \approx 0.032$).

In addition to higher average volatility, Cluster 1 also exhibits greater variability in behavior. The standard deviation of returns is approximately twice as large in Cluster 1 ($\text{logret\_21\_std} \approx 0.154$) compared to Cluster 0 ($\approx 0.078$), indicating more unstable price dynamics. Similarly, volatility variability ($\text{logvol\_21\_std}$) and volume fluctuation variability ($\text{vol\_chg\_21\_std}$) are substantially higher in Cluster 1.

These results confirm that Cluster 1 represents a more unstable and uncertain group of stocks, not only in terms of average levels but also in terms of variability. In contrast, Cluster 0 is characterized by more stable and consistent behavior across all dimensions.

These structural differences across clusters are expected to influence forecasting performance, as more stable stocks may exhibit more predictable volatility patterns.

## Cluster Interpretation

The clustering results reveal two distinct groups of stocks with different risk profiles. Cluster 0 represents lower-volatility stocks, characterized by stable returns, lower volatility levels, and moderate changes in trading volume. In contrast, Cluster 1 represents higher-volatility stocks, with slightly negative returns, higher volatility, and larger fluctuations in trading activity.

This separation confirms that the clustering approach successfully captures meaningful differences in stock behavior. The results align with financial intuition, where stocks can be broadly classified into more stable (defensive) and more volatile (risky) groups.

The sector composition of each cluster provides additional insight into the economic characteristics of the groups:

- Cluster 0 (lower-volatility group) is broadly diversified across sectors, with strong representation in Financials (70 stocks), Industrials (73 stocks), Health Care (53 stocks), and Information Technology (56 stocks). This suggests that more stable stocks are distributed across multiple sectors and are not concentrated in a single industry.

- Cluster 1 (higher-volatility group) is smaller and more concentrated. It has a relatively higher proportion of stocks in Information Technology (13 stocks) and Consumer Discretionary (10 stocks), sectors typically associated with higher growth potential but also greater uncertainty. Notably, sectors such as Consumer Staples, Real Estate, and Utilities have little to no representation in this cluster, which is consistent with their traditionally defensive and lower-risk nature.

Overall, the sector analysis reinforces the interpretation of the clusters. Lower-volatility stocks tend to be more diversified and include defensive sectors, while higher-volatility stocks are more concentrated in growth-oriented sectors that are inherently more sensitive to market conditions.

## Forecasting Performance

The performance of the LSTM model is evaluated against a naïve baseline model on the unseen test set.

The LSTM model achieved the following performance:

- RMSE: 0.0075  
- MAE: 0.0051  
- $R^2$: 0.338  
- Correlation: 0.5842  

In comparison, the naïve baseline produced:

- RMSE: 0.0104  
- MAE: 0.0069  
- $R^2$: -0.2746  
- Correlation: 0.4195  

The LSTM model significantly outperforms the baseline across all evaluation metrics. In particular, the reduction in RMSE and MAE indicates improved predictive accuracy, while the positive $R^2$ suggests that the model explains a meaningful portion of the variation in future volatility. The negative $R^2$ of the baseline highlights its inability to capture volatility dynamics beyond simple persistence.

## Training Behavior

Analysis of the training process shows that training loss decreases steadily over time, while validation loss remains relatively high. This indicates signs of early overfitting, suggesting that while the model learns patterns from the training data, its generalization ability is somewhat limited.

## Impact of Clustering on Forecasting

The inclusion of clustering information provides additional predictive value. The cluster-based feature allows the model to distinguish between different types of stocks, improving its ability to model volatility dynamics.

These results provide evidence that incorporating clustering improves forecasting performance by capturing structural differences across stocks.

However, the effectiveness of the model varies across clusters. The model performs better on the lower-volatility cluster, where behavior is more stable and predictable. In contrast, the higher-volatility cluster exhibits larger prediction errors and weaker model fit.

This suggests that while clustering improves forecasting performance overall, its benefits are more pronounced for stable stocks. Forecasting becomes significantly more challenging in high-volatility environments, where price movements are more erratic.

## Feature Importance

Feature importance analysis indicates that intraday price behavior plays a critical role in predicting volatility. In particular, the high–low range features (especially over the 21-day horizon) are the strongest predictors.

In addition, the cluster-based feature contributes meaningful predictive information, confirming that structural differences between stocks are relevant for forecasting. Rolling volatility and trading volume features also carry predictive signal, although to a lesser extent.

This suggests that volatility is driven not only by past returns but also by intraday price dynamics and trading behavior.

## Stock-Level Performance

At the individual stock level, model performance varies significantly. Only a minority of stocks achieve positive $R^2$, indicating that volatility remains difficult to predict at the individual stock level.

The best-performing stocks are primarily found in the lower-volatility cluster, suggesting that stable stocks are easier to model. In contrast, the most difficult-to-predict stocks appear in both clusters, reflecting the inherent complexity of financial markets.

## Summary of Findings

Overall, the results demonstrate that clustering provides meaningful structure in financial data and improves volatility forecasting performance when incorporated into predictive models. The LSTM model significantly outperforms the naïve baseline, and the inclusion of cluster information enhances predictive accuracy.

However, forecasting performance remains heterogeneous across stocks, with stronger results observed for lower-volatility assets. These findings highlight both the value and the limitations of combining clustering with time-series forecasting in financial applications.

# Conclusions

This study investigated whether clustering stocks based on their return, volatility, and trading activity characteristics improves the accuracy of stock volatility forecasting. The results provide clear evidence that clustering captures meaningful structure in financial data and contributes to improved predictive performance when incorporated into forecasting models.

The clustering analysis identified two distinct groups of stocks corresponding to lower-volatility and higher-volatility profiles. These clusters differ not only in average levels of returns and volatility, but also in their variability and sector composition. Lower-volatility stocks were found to be more stable and broadly distributed across sectors, while higher-volatility stocks exhibited greater instability and were more concentrated in growth-oriented industries. These findings confirm that financial markets exhibit heterogeneous behavior that can be effectively summarized through clustering techniques.

In the forecasting component, the LSTM model significantly outperformed the naïve baseline across all evaluation metrics. The results demonstrate that volatility is partially predictable when incorporating recent market behavior, including price dynamics, intraday variation, and trading activity. The inclusion of clustering information further improved model performance, indicating that structural differences between stocks provide additional predictive signal.

However, the results also highlight important limitations. Forecasting performance varied substantially across stocks, with stronger results observed for lower-volatility assets. In contrast, higher-volatility stocks remained difficult to predict, reflecting the inherently uncertain and unstable nature of financial markets. This suggests that while clustering enhances forecasting, it does not fully resolve the challenges associated with predicting extreme or irregular market behavior.

From a practical perspective, these findings suggest that clustering can be a valuable tool for improving risk modeling and portfolio management. Investors and practitioners may benefit from distinguishing between stable and high-risk assets when developing forecasting models or allocating capital. In particular, more reliable predictions can be obtained for lower-volatility stocks, while higher-volatility assets may require more robust or adaptive modeling approaches.

Alternative methods were also explored during the analysis. Hierarchical clustering was considered for the unsupervised learning component, and Random Forest models were tested for volatility forecasting. However, both approaches produced less stable clustering structures and lower predictive performance compared to the selected K-means and LSTM models, and were therefore not pursued further.

Overall, this study demonstrates that combining unsupervised learning techniques with time-series forecasting models provides a more comprehensive understanding of market dynamics. While the approach improves predictive performance, it also highlights the complexity of financial markets and the limitations of forecasting under uncertainty. Future research could explore more advanced clustering methods, alternative model architectures, or additional features to further enhance predictive accuracy.

# References
Clustering: Scikit-learn Developers. (2024). *KMeans clustering*. https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html
Forecast LSTM: TensorFlow Developers. (2024). *Long Short-Term Memory (LSTM) layer*. https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM
Forecast Naïve method: Hyndman, R. J., & Athanasopoulos, G. (2021). Forecasting: Principles and Practice (3rd ed.). OTexts. https://otexts.com/fpp3/
